In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from warnings import filterwarnings
filterwarnings ('ignore')

In [4]:
file_path = '/content/drive/MyDrive/Data Project/Covid/BehaviourDataset.csv'

In [5]:
df = pd.read_csv(file_path,encoding='UTF-16',sep='\t')
df.head(3)

,impact_on_academic_life,impact_on_mental_health,impact_on_social_life,Label
0,আমাদের শিক্ষাব্যবস্থা অনলাইন হওয়ার ক্ষেত্রে প...,লকডাউন এর কারণে দিন দিন ডিপ্রেশনের কারণ বেড়েই...,করোনার কারণে সামাজিক দূরত্ব অনেক বেড়ে যায়,Negative
1,অনলাইন শিক্ষাব্যবস্থা আমার কাছে খুবই চাপ যুক্ত...,অনলাইন শিক্ষা ব্যবস্থার ক্ষেত্রে আমি খুবই মানস...,অনলাইন শিক্ষা ব্যবস্থার ক্ষেত্রে আমি খুবই মানস...,Negative
2,অনলাইন থেকে আমার কাজের দক্ষতা বৃদ্ধির জন্য আমি...,পরিবারের সবার সাথে অনেক সময় কাটানো হয়েছে তাই...,পরিবার আত্মীয়-স্বজন এবং বন্ধু-বান্ধবের সাথে আ...,Positive


In [6]:
# =====================================================================
# BanglaBERT Fine-Tuning vs. Original Thesis Baselines
# Run this in Google Colab with GPU enabled:
# Runtime -> Change runtime type -> T4 GPU
# =====================================================================

# ---------------------------------------------------------------------
# STEP 0 — Install required libraries (Colab doesn't have these by default)
# ---------------------------------------------------------------------
!pip install -q transformers datasets scikit-learn accelerate

# ---------------------------------------------------------------------
# STEP 1 — Load the dataset
# ---------------------------------------------------------------------

df.columns = [c.strip() for c in df.columns]  # remove stray leading space in column names

print(f"Loaded {len(df)} rows")
print(df["Label"].value_counts())

# Convert Positive/Negative to 1/0, same as the original paper's label encoding
df["label_encoded"] = df["Label"].map({"Positive": 0, "Negative": 1})

TEXT_COLUMNS = {
    "academic": "impact_on_academic_life",
    "mental": "impact_on_mental_health",
    "social": "impact_on_social_life",
}

# Baselines from the original thesis (Tables 5 and 6), for the final comparison
ORIGINAL_BASELINES = {
    "academic": {"best_ml": ("SGD", 95.00), "best_dl": ("BiLSTM", 92.50)},
    "mental":   {"best_ml": ("KNN", 93.75), "best_dl": ("LSTM", 98.75)},
    "social":   {"best_ml": ("SGD/Multi-NB", 95.00), "best_dl": ("BiLSTM/CNN/CNN-LSTM", 92.50)},
}

# ---------------------------------------------------------------------
# STEP 2 — Imports for fine-tuning
# ---------------------------------------------------------------------
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

MODEL_NAME = "sagorsarker/bangla-bert-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }


def fine_tune_on_column(column_key, text_column):
    print(f"\n{'='*60}\nFine-tuning on: {column_key} ({text_column})\n{'='*60}")

    texts = df[text_column].astype(str).tolist()
    labels = df["label_encoded"].tolist()

    # Same 80/20 split logic as the original paper, stratified to keep
    # the class balance even in the smaller test set
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=42
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize(batch):
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

    train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(tokenize, batched=True)
    test_ds = Dataset.from_dict({"text": test_texts, "label": test_labels}).map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    args = TrainingArguments(
        output_dir=f"./results_{column_key}",
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=5,
        learning_rate=2e-5,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    return results


# ---------------------------------------------------------------------
# STEP 3 — Run fine-tuning for all three columns and collect results
# ---------------------------------------------------------------------
all_results = {}
for key, col in TEXT_COLUMNS.items():
    all_results[key] = fine_tune_on_column(key, col)

# ---------------------------------------------------------------------
# STEP 4 — Final comparison table: BanglaBERT vs. original thesis results
# ---------------------------------------------------------------------
print("\n\n" + "=" * 70)
print("FINAL COMPARISON: BanglaBERT vs. Original Thesis Results")
print("=" * 70)

comparison_rows = []
for key in TEXT_COLUMNS:
    bert_acc = all_results[key]["eval_accuracy"] * 100
    bert_f1 = all_results[key]["eval_f1"] * 100
    best_ml_name, best_ml_acc = ORIGINAL_BASELINES[key]["best_ml"]
    best_dl_name, best_dl_acc = ORIGINAL_BASELINES[key]["best_dl"]

    comparison_rows.append({
        "Column": key,
        "BanglaBERT Accuracy (%)": round(bert_acc, 2),
        "BanglaBERT F1 (%)": round(bert_f1, 2),
        f"Best ML ({best_ml_name})": best_ml_acc,
        f"Best DL ({best_dl_name})": best_dl_acc,
        "BanglaBERT beats both?": "Yes" if bert_acc > max(best_ml_acc, best_dl_acc) else "No",
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

comparison_df.to_csv("/content/drive/MyDrive/Data Project/Covid/banglabert_vs_thesis_comparison.csv", index=False)
print("\nSaved comparison table to banglabert_vs_thesis_comparison.csv")

Loaded 400 rows
Label
Negative    200
Positive    200
Name: count, dtype: int64
Using device: cuda

Fine-tuning on: academic (impact_on_academic_life)


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.364586,0.288161,0.912500,0.915663,0.883721,0.950000
2,0.261968,0.225556,0.925000,0.926829,0.904762,0.950000
3,0.066487,0.222410,0.925000,0.928571,0.886364,0.975000
4,0.121262,0.215420,0.937500,0.938272,0.926829,0.950000


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.121262,0.215420,4,0.937500,0.938272,0.926829,0.950000




FINAL COMPARISON: BanglaBERT vs. Original Thesis Results
  Column  BanglaBERT Accuracy (%)  BanglaBERT F1 (%)  Best ML (SGD)  Best DL (BiLSTM) BanglaBERT beats both?  Best ML (KNN)  Best DL (LSTM)  Best ML (SGD/Multi-NB)  Best DL (BiLSTM/CNN/CNN-LSTM)
academic                    87.50              86.11           95.0              92.5                     No            NaN             NaN                     NaN                            NaN
  mental                    92.50              92.68            NaN               NaN                     No          93.75           98.75                     NaN                            NaN
  social                    93.75              93.83            NaN               NaN                     No            NaN             NaN                    95.0                           92.5

Saved comparison table to banglabert_vs_thesis_comparison.csv


In [ ]:
# =====================================================================
# BanglaBERT Fine-Tuning vs. Original Thesis Baselines — 5-Fold CV
# Run this in Google Colab with GPU enabled:
# Runtime -> Change runtime type -> T4 GPU
#
# Difference from the single-split version: each column is evaluated
# with 5-fold cross-validation instead of one 80/20 split, so the
# result is a mean +/- standard deviation rather than a single number
# that could swing on just a handful of examples (the test set is
# only 80 rows, so a single split is fragile).
# =====================================================================

!pip install -q transformers datasets scikit-learn accelerate

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

# ---------------------------------------------------------------------
# STEP 1 — Load the dataset
# ---------------------------------------------------------------------

df.columns = [c.strip() for c in df.columns]
df["label_encoded"] = df["Label"].map({"Positive": 0, "Negative": 1})

TEXT_COLUMNS = {
    "academic": "impact_on_academic_life",
    "mental": "impact_on_mental_health",
    "social": "impact_on_social_life",
}

ORIGINAL_BASELINES = {
    "academic": {"best_ml": ("SGD", 95.00), "best_dl": ("BiLSTM", 92.50)},
    "mental":   {"best_ml": ("KNN", 93.75), "best_dl": ("LSTM", 98.75)},
    "social":   {"best_ml": ("SGD/Multi-NB", 95.00), "best_dl": ("BiLSTM/CNN/CNN-LSTM", 92.50)},
}

MODEL_NAME = "sagorsarker/bangla-bert-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

N_FOLDS = 5


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }


def tokenize_batch(tokenizer, texts):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=64)


def cross_validate_column(column_key, text_column):
    print(f"\n{'='*60}\n5-Fold CV on: {column_key} ({text_column})\n{'='*60}")

    texts = df[text_column].astype(str).tolist()
    labels = df["label_encoded"].to_numpy()
    texts = np.array(texts)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(texts, labels), start=1):
        print(f"\n--- Fold {fold_idx}/{N_FOLDS} ---")

        train_texts, val_texts = texts[train_idx].tolist(), texts[val_idx].tolist()
        train_labels, val_labels = labels[train_idx].tolist(), labels[val_idx].tolist()

        train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(
            lambda b: tokenize_batch(tokenizer, b["text"]), batched=True
        )
        val_ds = Dataset.from_dict({"text": val_texts, "label": val_labels}).map(
            lambda b: tokenize_batch(tokenizer, b["text"]), batched=True
        )

        # Fresh model each fold — reusing a trained model across folds
        # would leak information between folds
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

        args = TrainingArguments(
            output_dir=f"./results_{column_key}_fold{fold_idx}",
            num_train_epochs=4,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            eval_strategy="epoch",
            save_strategy="no",
            logging_steps=50,
            learning_rate=2e-5,
            report_to="none",
            disable_tqdm=True,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        fold_metrics = trainer.evaluate()
        fold_results.append(fold_metrics)

        print(f"Fold {fold_idx} accuracy: {fold_metrics['eval_accuracy']*100:.2f}%")

        # free GPU memory before the next fold
        del model, trainer
        torch.cuda.empty_cache()

    accs = [r["eval_accuracy"] * 100 for r in fold_results]
    f1s = [r["eval_f1"] * 100 for r in fold_results]

    return {
        "mean_accuracy": np.mean(accs),
        "std_accuracy": np.std(accs),
        "mean_f1": np.mean(f1s),
        "std_f1": np.std(f1s),
        "fold_accuracies": accs,
    }


# ---------------------------------------------------------------------
# STEP 2 — Run 5-fold CV for all three columns
# ---------------------------------------------------------------------
all_results = {}
for key, col in TEXT_COLUMNS.items():
    all_results[key] = cross_validate_column(key, col)

# ---------------------------------------------------------------------
# STEP 3 — Final comparison table
# ---------------------------------------------------------------------
print("\n\n" + "=" * 70)
print("FINAL COMPARISON: BanglaBERT (5-fold CV) vs. Original Thesis Results")
print("=" * 70)

comparison_rows = []
for key in TEXT_COLUMNS:
    r = all_results[key]
    best_ml_name, best_ml_acc = ORIGINAL_BASELINES[key]["best_ml"]
    best_dl_name, best_dl_acc = ORIGINAL_BASELINES[key]["best_dl"]

    comparison_rows.append({
        "Column": key,
        "BanglaBERT Acc (mean ± std, %)": f"{r['mean_accuracy']:.2f} ± {r['std_accuracy']:.2f}",
        "BanglaBERT F1 (mean, %)": round(r["mean_f1"], 2),
        f"Best ML ({best_ml_name})": best_ml_acc,
        f"Best DL ({best_dl_name})": best_dl_acc,
        "Fold accuracies": [round(a, 1) for a in r["fold_accuracies"]],
    })

comparison_df = pd.DataFrame(comparison_rows)
pd.set_option("display.width", 150)
print(comparison_df.to_string(index=False))

comparison_df.to_csv("banglabert_cv_vs_thesis_comparison.csv", index=False)
print("\nSaved to banglabert_cv_vs_thesis_comparison.csv")

Using device: cuda

5-Fold CV on: academic (impact_on_academic_life)

--- Fold 1/5 ---


Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.3704', 'eval_accuracy': '0.85', 'eval_f1': '0.8378', 'eval_precision': '0.9118', 'eval_recall': '0.775', 'eval_runtime': '0.36', 'eval_samples_per_second': '222.2', 'eval_steps_per_second': '13.89', 'epoch': '1'}
{'eval_loss': '0.2915', 'eval_accuracy': '0.8625', 'eval_f1': '0.8608', 'eval_precision': '0.8718', 'eval_recall': '0.85', 'eval_runtime': '0.361', 'eval_samples_per_second': '221.6', 'eval_steps_per_second': '13.85', 'epoch': '2'}
{'loss': '0.3113', 'grad_norm': '7.597', 'learning_rate': '7.75e-06', 'epoch': '2.5'}
{'eval_loss': '0.2905', 'eval_accuracy': '0.8875', 'eval_f1': '0.8861', 'eval_precision': '0.8974', 'eval_recall': '0.875', 'eval_runtime': '0.3667', 'eval_samples_per_second': '218.2', 'eval_steps_per_second': '13.64', 'epoch': '3'}
{'eval_loss': '0.3219', 'eval_accuracy': '0.875', 'eval_f1': '0.8718', 'eval_precision': '0.8947', 'eval_recall': '0.85', 'eval_runtime': '0.3673', 'eval_samples_per_second': '217.8', 'eval_steps_per_second': '13.61', 